# Chapter 11: Batch Processing
*From: Designing Data-Intensive Applications*

## TL;DR

- **Batch processing** takes immutable, bounded input datasets and produces fresh output files from scratch — no in-place mutation, no side effects. Jobs can be rerun after a code fix and the output is regenerated correctly (*human fault tolerance*).
- The **immutable-input / derived-output** property is what gives batch its superpowers: easy rollback, multiple consumers of the same inputs, and cheap experimentation.
- Every batch system has the same **three layers**: a **storage layer** (distributed filesystem or object store), an **orchestration layer** (cluster resource manager + workflow scheduler), and a **compute layer** (MapReduce, Spark, Flink, or SQL engine).
- **MapReduce** is the canonical but largely obsolete model (map → sort/shuffle → reduce, always materialize intermediates to DFS). **Dataflow engines** (Spark, Flink) replace map/reduce alternation with DAGs of operators that pipeline in memory and reuse JVMs — much faster.
- Batch output rarely stays in the DFS: it flows to **ETL warehouses, analytics lakehouses, ML training, and derived-data serving** — the last usually via Kafka or by bulk-loading prebuilt database files, never by writing one record at a time from inside tasks.

## Why Batch Processing (and when to use it)

Most of the book so far has been about **online systems** — request/response, latency-sensitive, always-on. Batch is the opposite regime:

- **Input is bounded and already at rest.** A daily dump of the logs, a snapshot of the user table, a new shard of web-crawl HTML.
- **Throughput, not latency, is the metric.** A job may run for minutes, hours, or days. Nobody is waiting on a single record.
- **Output is fully regenerated every run.** If the job produced wrong output yesterday, fix the code and rerun today — the old output is either versioned (object store "time travel") or overwritten.

Where batch still dominates, straight from the chapter:

- **Accounting and inventory reconciliation** — verify transactions against bank/inventory snapshots overnight.
- **Demand forecasting** in manufacturing and retail (Getir runs demand forecasts as periodic batch jobs).
- **Recommendation model training** at ecommerce, media, and social-media companies.
- **The US banking network (ACH)** runs almost entirely on batch jobs.
- **LLM training data preparation** — extracting text from HTML, dedup, tokenization — is embarrassingly parallel batch work (OpenAI uses Ray for this).

Batch is the wrong tool when you need fresh results in seconds — that's *stream processing* (Chapter 12) — or when each answer depends on one user's request (that's an online system).

## The Unix Tools Blueprint

Before distributed anything, Unix already had a batch-processing pipeline. Here is the canonical *top-5 URLs* example from the chapter:

```shell
cat /var/log/nginx/access.log |
  awk '{print $7}' |
  sort |
  uniq -c |
  sort -r -n |
  head -n 5
```

Each stage is a small streaming program connected by pipes; no stage holds the full dataset.

```mermaid
flowchart LR
  A[access.log] --> B[cat: stream bytes]
  B --> C["awk '{print $7}'<br/>mapper: extract URL"]
  C --> D[sort: group equal keys]
  D --> E["uniq -c<br/>reducer: count runs"]
  E --> F[sort -r -n: rank by count]
  F --> G[head -n 5: top-5]
```

The equivalent Python keeps a `defaultdict(int)` of counts in memory — simpler to read, but it only works while the **working set** (distinct URLs) fits in RAM.

**The sort-vs-hash choice is the central insight.** It reappears at every scale:

| Strategy | When it wins | What limits it | Disk behavior |
|----------|--------------|----------------|---------------|
| **In-memory hash aggregation** (Python dict) | Working set (distinct keys) fits in RAM | RAM size, not dataset size | No spill; one big hot map |
| **External-merge sort** (Unix `sort`, MapReduce shuffle) | Working set larger than memory | Disk + network bandwidth | Spill sorted runs, k-way merge |

GNU `sort` already spills to disk and parallelizes across cores; MapReduce and Spark generalize the same algorithm to a cluster. The Unix pipeline is the **same program** — just on one machine.

The Unix example also shows the other half of batch's value: **immutable inputs + regenerated outputs**. Delete `output.log`, rerun; if `awk` had a bug, fix it and rerun. You can't do that with a read/write database — writing bad data is permanent damage.

## Architecture Layers

A distributed batch framework is, literally, a distributed operating system: same three layers you have on a laptop, just scaled out.

```mermaid
flowchart TB
  subgraph CMP["Compute Layer (programs)"]
    MR[MapReduce]
    SP[Spark / Flink dataflow]
    SQL[SQL engines<br/>Trino / BigQuery / Snowflake]
    DF[DataFrame APIs<br/>Pandas / Spark DF / Daft]
  end

  subgraph ORCH["Orchestration Layer (the kernel)"]
    WF[Workflow Scheduler<br/>Airflow / Dagster / Prefect<br/>DAGs of jobs]
    RM[Resource Manager<br/>YARN / Kubernetes<br/>per-job scheduling]
    EX[Task Executors<br/>NodeManager / kubelet<br/>cgroups isolation]
  end

  subgraph STG["Storage Layer (the filesystem)"]
    DFS[Distributed Filesystem<br/>HDFS / JuiceFS / 3FS<br/>128 MB blocks, data-local]
    OS[Object Store<br/>S3 / GCS / Azure Blob<br/>immutable keyed objects]
  end

  CMP --> ORCH
  ORCH --> STG
  WF --> RM
  RM --> EX
```

The arrows matter: compute reads and writes to storage; orchestration decides *where* each task runs and waits for upstream jobs to finish before firing downstream ones.

## Deep Dive 1: Unix Tools as Batch Foundation

Unix got the design right in the 1970s. A batch pipeline is just a chain of small programs where:

1. **Inputs are files** (read-only during the job).
2. **Each stage is stateless, streaming** — reads stdin, writes stdout.
3. **The shell (or scheduler) wires them together** with pipes.
4. **Outputs are new files**, never in-place mutations.

Every distributed framework keeps these four properties; the only things it adds are *sharding*, *fault tolerance*, and *cluster scheduling*. The single biggest engineering win is already present in Unix: **immutable inputs + regenerated outputs** let you rerun on bugs, switch between versions (old output in another directory), and compare runs for monitoring.

See: [[01-unix-tools-as-batch-foundation]]

## Deep Dive 2: Distributed Storage — DFS vs Object Stores

Storage is the first thing batch frameworks need. Two families:

- **Distributed Filesystems** — HDFS, JuiceFS, 3FS, GlusterFS, CephFS. Files split into large blocks (HDFS defaults to **128 MB** vs ext4's 4,096 bytes), replicated 3x across **DataNodes**, metadata held by a **NameNode** (or a metadata service backed by FoundationDB in 3FS). Tasks are scheduled *where a replica lives* to minimize network I/O.
- **Object Stores** — S3, GCS, Azure Blob, R2, MinIO, B2. Each object has a URL (`s3://bucket/key`); keys are flat; "directories" are just prefix conventions. Objects are **immutable**: updates require a full `put`. Rename is nonatomic (copy + delete). Storage and compute are **decoupled** — you scale them independently.

```mermaid
flowchart LR
  subgraph HDFS["HDFS cluster (data-local)"]
    NN[NameNode<br/>metadata: blocks→nodes]
    DN1[DataNode 1<br/>blocks A1,B1,C1]
    DN2[DataNode 2<br/>blocks A2,B2,C2]
    DN3[DataNode 3<br/>blocks A3,B3,C3]
    NN -.metadata.-> DN1
    NN -.metadata.-> DN2
    NN -.metadata.-> DN3
  end
  T1[Map Task for block A] -->|scheduled on| DN1
  T1 -.reads locally.-> DN1
```

Most modern pipelines have migrated to object stores despite losing data locality — datacenter networks are fast enough, and independent scaling is a bigger win.

| Dimension | DFS (HDFS) | Object Store (S3) |
|-----------|-----------|-------------------|
| Block size | ~128 MB | 4 MB chunks, whole-object API |
| Mutability | Mutable append | Immutable; rewrite on update |
| Directory semantics | Real directories | Prefix convention only |
| Rename | Atomic | Copy + delete (nonatomic) |
| Data locality | Yes (task near replica) | No (compute is disaggregated) |
| POSIX support | Often (FUSE/NFS) | Rare |
| Scaling model | Storage + compute coupled | Storage and compute scale independently |

**Redundancy.** Both families replicate, but with different math:

- **3x replication** — simple, fast recovery, 200% overhead.
- **Reed–Solomon erasure coding** — e.g., 10 data chunks + 4 parity chunks — lower storage overhead but higher reconstruction cost (must read k chunks to rebuild one lost one).

### Storage-layer capacity math

Two quick calculations from the chapter: (a) metadata overhead from block size and (b) storage overhead of replication vs erasure coding.

In [ ]:
# (a) Block size matters for metadata overhead.
# ext4 uses 4 KiB blocks; HDFS uses 128 MiB blocks.
# For a 900 MB file (from the chapter), how many blocks each?

file_size_mb = 900
ext4_block_bytes = 4096
hdfs_block_bytes = 128 * 1024 * 1024  # 128 MiB

file_bytes = file_size_mb * 1024 * 1024

ext4_blocks = -(-file_bytes // ext4_block_bytes)          # ceil division
hdfs_full_blocks = file_bytes // hdfs_block_bytes
hdfs_tail_bytes = file_bytes - hdfs_full_blocks * hdfs_block_bytes
hdfs_total_blocks = hdfs_full_blocks + (1 if hdfs_tail_bytes else 0)

print(f"900 MB file:")
print(f"  on ext4 (4 KiB blocks)  : {ext4_blocks:>10,} blocks of metadata to track")
print(f"  on HDFS (128 MiB blocks): {hdfs_total_blocks:>10,} blocks "
      f"({hdfs_full_blocks} full of 128 MiB + 1 tail of {hdfs_tail_bytes/1024/1024:.1f} MiB)")
print(f"  HDFS metadata is ~{ext4_blocks // hdfs_total_blocks:,}x smaller.")
print()

# (b) 3x replication vs Reed-Solomon (10+4) erasure coding.
raw_pb = 1.0  # 1 PB of logical data

replication_overhead = 3.0                 # store 3x
rs_data, rs_parity = 10, 4                 # 10 data + 4 parity chunks
rs_overhead = (rs_data + rs_parity) / rs_data  # = 1.4x

print(f"Storing {raw_pb} PB of logical data:")
print(f"  3x replication        : {raw_pb * replication_overhead:.2f} PB physical "
      f"(+{(replication_overhead - 1) * 100:.0f}% overhead)")
print(f"  RS(10,4) erasure code : {raw_pb * rs_overhead:.2f} PB physical "
      f"(+{(rs_overhead - 1) * 100:.0f}% overhead)")
print(f"  Savings at 1 PB       : {(replication_overhead - rs_overhead) * raw_pb:.2f} PB")
print(f"  Tradeoff: RS needs {rs_data} chunk reads to reconstruct 1 lost chunk "
      f"(replication needs just 1).")

See: [[02-distributed-storage-for-batch]]

## Deep Dive 3: Distributed Job Orchestration

Once data is in the DFS, you need something to *run programs* against it. That "something" has two distinct layers — often conflated, but do different things:

**Cluster orchestrator** (YARN, Kubernetes): a per-*job* scheduler that manages CPU, memory, GPUs, and disks across a cluster.

```mermaid
flowchart TB
  Client[Batch Job Request<br/>10 tasks × 4 GB × 2 cores] --> Sched
  subgraph Orchestrator
    Sched[Scheduler<br/>FIFO / DRF / bin-packing]
    RM[Resource Manager<br/>cluster state in ZooKeeper/etcd]
    Sched <--> RM
  end
  RM -.node state.-> N1
  RM -.node state.-> N2
  RM -.node state.-> N3
  Sched -->|dispatch tasks| N1[Executor on Node 1<br/>NodeManager/kubelet]
  Sched -->|dispatch tasks| N2[Executor on Node 2]
  Sched -->|dispatch tasks| N3[Executor on Node 3]
```

Components:

- **Task executors** (NodeManager / kubelet) — run on every node, launch task processes, enforce isolation with Linux cgroups, send heartbeats.
- **Resource manager** — stores node state (hardware, task status, health) in a coordination service (ZooKeeper for YARN, etcd for K8s).
- **Scheduler** — decides *which task runs where*. Provably NP-hard in the general case, so it uses heuristics: FIFO, dominant resource fairness (DRF), priority queues, bin-packing, gang scheduling.

**The scheduler's dilemma** (the chapter's 160-core thought experiment):

| Strategy | Effect on utilization | Risk |
|----------|-----------------------|------|
| **Gang scheduling** (reserve all 100 cores before starting) | Nodes sit idle while reserving → low utilization | Deadlock if multiple jobs reserve |
| **Incremental allocation** (start with what's free, grow) | High utilization | *Starvation* — job may never get all 100 cores |
| **Preemption** (kill low-priority tasks) | Frees cores fast | Killed work has to be redone |

**Spot / preemptible instances** turn this into an economic lever — low-priority, cheap compute that tolerates getting killed is perfect for batch, because the framework already retries failed tasks.

**Workflow scheduler** (Airflow, Dagster, Prefect): a per-*DAG* scheduler that sits *above* the cluster orchestrator. It waits for upstream job outputs to exist in the DFS/object store, then fires the downstream job. Modern pipelines have 50–100 jobs in a DAG; workflow schedulers are what make this manageable.

The split: **cluster orchestrator = per-job resources; workflow scheduler = per-DAG dependencies.**

See: [[03-distributed-job-orchestration]]

## Deep Dive 4: MapReduce

MapReduce is the 2004 Google paper that started all of this. The model is deliberately narrow — just two user callbacks:

- **Mapper** — called once per input record, emits zero or more `(key, value)` pairs. Stateless across records.
- **Reducer** — receives all values sharing a key, emits output records.

Connecting the two is a framework-managed **shuffle** (covered in the next section) that sorts and groups intermediate data.

```mermaid
flowchart LR
  I1[Input shard 1] --> M1[Mapper 1]
  I2[Input shard 2] --> M2[Mapper 2]
  I3[Input shard 3] --> M3[Mapper 3]
  M1 --> S[Shuffle<br/>hash-partition + sort by key]
  M2 --> S
  M3 --> S
  S --> R1[Reducer 1]
  S --> R2[Reducer 2]
  S --> R3[Reducer 3]
  R1 --> O1[Output r1]
  R2 --> O2[Output r2]
  R3 --> O3[Output r3]
```

Tie this back to the Unix example: `awk '{print $7}'` is the mapper, the first `sort` is the implicit shuffle, `uniq -c` is the reducer.

**Why MapReduce won in the 2000s — and why it lost.**

| Property | Consequence |
|----------|-------------|
| Stateless mapper/reducer (functional-programming heritage) | Deterministic replay → safe to retry any task |
| Every intermediate written to DFS (replicated 3x) | Survives any number of node failures → preemption-friendly |
| Map and reduce are the *only* operators | Any join, filter, or windowing must be hand-coded |
| New JVM per task | Huge startup overhead |
| File-based I/O between stages | Prevents pipelining — each stage blocks on the full previous stage |

The chapter is blunt: "[MapReduce] is now largely obsolete and no longer used at Google." Dataflow engines preserved the good parts (stateless user functions, task-level retries) and fixed the bad parts (always-materialize-to-DFS, strict map→reduce alternation).

See: [[04-mapreduce]]

## Deep Dive 5: Dataflow Engines — Spark, Flink, and High-Level APIs

Dataflow engines model a whole workflow as **one DAG of operators**, not a chain of independent map/reduce jobs. Operators include `filter`, `map`, `join`, `groupBy`, `aggregate`.

Key optimizations over MapReduce:

- **Pipelining.** Operators that don't reshard (map, filter) fuse into a single task — no serialization, no disk writes between them.
- **Sort only when needed.** MapReduce sorts between every map and reduce; dataflow engines sort only where the plan demands it (e.g., a sort-merge join).
- **Intermediate state in memory or local disk.** No 3x-replicated DFS writes for every stage boundary.
- **Locality-aware consumer placement.** The scheduler knows the full DAG, so it co-locates consumer tasks with their producer tasks — data travels through shared memory instead of over the network.
- **JVM reuse.** Long-lived executors run many tasks, so startup cost is amortized.
- **High-level APIs** on top: SQL (with cost-based optimizers), DataFrames (Spark, Daft, Pandas-like), graph APIs (GraphX, Gelly, Pregel/BSP).

**Two answers to "how do we recover lost intermediate state without DFS writes?"**

| Engine | Recovery mechanism |
|--------|-------------------|
| **Spark** | RDD lineage — remember *how* each partition was computed; on loss, recompute from upstream |
| **Flink** | Periodic checkpointing — snapshot task state; on loss, restart from last checkpoint |

```mermaid
flowchart LR
  subgraph Stage1[Stage 1 - pipelined on node A]
    R[Read Parquet] --> F[Filter: status=200] --> M[Map: extract URL,user_id]
  end
  subgraph Stage2[Stage 2 - pipelined on node B]
    J[Join with user_dim] --> G[Group by URL] --> A2[Agg: count]
  end
  subgraph Stage3[Stage 3]
    W[Write output]
  end
  Stage1 -->|shuffle boundary| Stage2
  Stage2 --> Stage3
```

**Why high-level APIs matter.** A handwritten MapReduce join is tedious and error-prone. A SQL query goes through a **cost-based optimizer** that can reorder joins, pick between sort-merge and broadcast-hash joins, and exploit columnar storage (Parquet) for pushdown. The same is true for DataFrames — Spark defers execution, optimizes, and runs the plan.

See: [[05-dataflow-engines-spark-flink]]

## Deep Dive 6: Shuffle and Distributed Joins

**Shuffle** is the single most important distributed-batch primitive. It routes every `(key, value)` pair emitted by any mapper to the reducer responsible for its hashed key, so all values for a key arrive at the same reducer, **sorted**.

```mermaid
flowchart TB
  subgraph Mappers
    M1[Mapper 1]
    M2[Mapper 2]
    M3[Mapper 3]
  end
  subgraph Spills["Per-reducer spill files on mapper's local disk"]
    M1 --> m1r1[m1,r1] & m1r2[m1,r2] & m1r3[m1,r3]
    M2 --> m2r1[m2,r1] & m2r2[m2,r2] & m2r3[m2,r3]
    M3 --> m3r1[m3,r1] & m3r2[m3,r2] & m3r3[m3,r3]
  end
  m1r1 --> R1[Reducer 1<br/>merge + iterate by key]
  m2r1 --> R1
  m3r1 --> R1
  m1r2 --> R2[Reducer 2]
  m2r2 --> R2
  m3r2 --> R2
  m1r3 --> R3[Reducer 3]
  m2r3 --> R3
  m3r3 --> R3
```

Mechanics:

1. Each mapper bucketizes its output by `hash(key) mod num_reducers`, writing **one sorted spill file per reducer** to local disk.
2. Reducers pull their assigned bucket from every mapper (network fetch).
3. Each reducer **merge-sorts** its incoming sorted streams — the key values for a given key are now adjacent, regardless of which mapper emitted them.
4. The reducer function is called once per key with an iterator over its values.

### Shuffle capacity math

For an M × R shuffle, the number of intermediate spill files is M × R — this grows fast.

In [ ]:
# Shuffle intermediate data volume
# Consider a Spark/MapReduce job:
#   - 500 mappers (each processing a ~128 MB block of input)
#   - 200 reducers
#   - each mapper emits roughly 100 MB of intermediate data total

num_mappers = 500
num_reducers = 200
intermediate_per_mapper_mb = 100

# Each mapper writes one spill file per reducer -> M*R total files.
total_spill_files = num_mappers * num_reducers
bytes_per_spill_mb = intermediate_per_mapper_mb / num_reducers

total_intermediate_mb = num_mappers * intermediate_per_mapper_mb
total_intermediate_gb = total_intermediate_mb / 1024

print(f"Mappers: {num_mappers:,}  |  Reducers: {num_reducers:,}")
print(f"Per-reducer spill files:   {total_spill_files:>10,} "
      f"({bytes_per_spill_mb:.2f} MB each)")
print(f"Total intermediate shuffle: {total_intermediate_gb:>10,.1f} GB")
print(f"Network moved in shuffle:  {total_intermediate_gb:>10,.1f} GB "
      f"(every byte crosses the network once)")
print()
print("This is why MapReduce writes all of this to the DFS (safe, slow)")
print("and why Spark/Flink keep it in RAM or local disk (fast, recomputable).")

**Sort-merge join** falls out of shuffle almost for free. Suppose we want to join an `activity_events` log with a `user_profiles` table on `user_id`:

```mermaid
flowchart LR
  AE[activity_events<br/>mapper emits user_id -> event] --> S[Shuffle by user_id<br/>secondary sort:<br/>profile first, then events by ts]
  UP[user_profiles<br/>mapper emits user_id -> profile] --> S
  S --> R[Reducer for each user_id:<br/>1st record = profile<br/>following = events<br/>emit enriched event]
```

The reducer keeps *one* profile in memory at a time — no network, no random lookups. **Secondary sort** orders the reducer's input so the profile arrives before the events.

**Join algorithm choice.** Cost-based optimizers pick automatically:

| Join type | How it works | When it wins |
|-----------|--------------|--------------|
| **Sort-merge (shuffle) join** | Shuffle both sides by the join key, merge | Both sides large; scales to petabytes |
| **Broadcast hash join** | Ship the small table to every mapper, build a hash map | One side fits in memory (small dimension table) |

See: [[06-shuffle-and-distributed-joins]]

## Deep Dive 7: Serving Derived Data from Batch

Batch is rarely the end of the pipeline — the output has to make it to live traffic. The chapter's four big use-case families:

| Use case | Typical tools | What's being built |
|----------|---------------|--------------------|
| **ETL / ELT** | Airflow + Spark/Trino/DuckDB + Snowflake/BigQuery | Extract from OLTP, transform, load into warehouse |
| **Analytics (lakehouse)** | Iceberg tables + Unity/Hive Metastore + SparkSQL/Trino + BI tools | Ad-hoc + pre-aggregated OLAP over DFS/object-store data |
| **Machine learning** | Spark MLlib, Flink ML, Ray, Kubeflow, Flyte | Feature engineering, model training, batch inference, LLM preprocessing |
| **Derived data for serving** | Batch → Kafka / bulk SST files → Elasticsearch, Druid, Pinot, Venice | Recommendations, search indexes, feature stores |

The last one is worth a diagram — it's where batch touches user-facing production.

```mermaid
flowchart LR
  B[Batch Job<br/>Spark/Flink] --> O[Output files<br/>DFS or Object Store]
  O --> K[Kafka topic<br/>buffered, replayable]
  K --> ES[Elasticsearch<br/>search index]
  K --> DR[Druid / Pinot<br/>real-time OLAP]
  K --> VN[Venice<br/>derived-data KV store]
  K --> CH[ClickHouse<br/>analytics]
  O -.bulk file swap.-> PIN[Pinot via Hadoop import]
  O -.bulk SST files.-> RDB[RocksDB bulk-import]
  O -.Lightning import.-> TDB[TiDB]
  N[Completion notification] -.gates visibility.-> ES & DR & VN
```

**The anti-pattern** (called out explicitly in the chapter): having parallel batch tasks write **one record at a time** directly to a production database. Problems:

1. Per-record network round-trips are orders of magnitude slower than batch throughput.
2. Thousands of parallel tasks overwhelm the production DB, hurting user queries.
3. Partial failures leak — batch's clean "all-or-nothing" guarantee is broken because side effects are already visible externally.
4. Task retries duplicate output.

**Two correct alternatives:**

| Approach | Pros | Cons |
|----------|------|------|
| **Stream through Kafka** | Sequential writes, buffered, multi-consumer, DMZ-friendly | Need completion notifications to gate visibility (read-committed-style) |
| **Bulk-load prebuilt files** (SST, Lightning, Pinot import) | Atomic version swap, no per-record overhead | Harder to do incremental updates; "rebuild the world" every time |

Venice's hybrid mode (batch full-rebuild + stream incrementals) is the common real-world answer.

See: [[07-serving-derived-data-from-batch]]

## Trade-offs: Batch Compute Engines Compared

| Engine | Programming model | Fault tolerance | Typical use | Performance |
|--------|-------------------|-----------------|-------------|-------------|
| **MapReduce** | Two callbacks (map, reduce); every join hand-coded | Intermediates always written to DFS (3x replicated) — very robust, very slow | Legacy Hadoop workloads; largely replaced | Slow — file I/O between every stage, new JVM per task |
| **Dataflow engines (Spark, Flink)** | DAG of operators; RDDs / DataStreams; SQL and DataFrames on top | Spark: RDD lineage recomputation. Flink: periodic checkpoints | Modern general-purpose batch + ML; complex DAGs; graph/BSP | Fast — pipelined, in-memory, JVM reuse, locality-aware |
| **SQL query engines (Trino, Hive, SparkSQL, BigQuery)** | Declarative SQL; cost-based optimizer picks plan | Engine-dependent; typically task-retry + shuffle services | Interactive analytics, lakehouse queries, ELT transforms | Very fast on columnar (Parquet/Arrow); good for relational workloads only |
| **Cloud data warehouses (Snowflake, BigQuery)** | SQL + DataFrame libraries (Snowpark, BQ DataFrames) | Managed — transparent to user | Managed analytics, BI, scheduled pre-aggregation | Very fast on relational; expensive for large custom ML/ETL |

**Rules of thumb from the chapter:**

- Reach for **dataflow engines** when your workflow is complex, has multiple stages, or needs iterative algorithms.
- Reach for a **cloud data warehouse** when the work is SQL-expressible and developer time matters more than cost.
- Reach for a dedicated **batch framework (Spark/Flink/Ray)** when you have non-SQL work — ML training, LLM preprocessing, graph algorithms, multimodal data — or when cloud-warehouse costs dominate.

## Key Takeaways

1. **Batch processing is defined by immutable inputs and regenerated outputs** — that single property gives you human fault tolerance, easy rollback, multi-consumer reuse, and cheap experimentation. Everything else is an implementation detail.
2. **Every batch framework is a distributed operating system** with the same three layers: storage (DFS/object store), orchestration (resource manager + workflow scheduler), and compute (MapReduce/dataflow/SQL).
3. **The sort-vs-hash choice** from the Unix `awk | sort | uniq` pipeline is the same choice distributed shuffles make: in-memory aggregation when the working set fits, external-merge sort when it doesn't.
4. **MapReduce's rigid "always write intermediates to DFS" fault-tolerance model** was its strength in a preemption-heavy world and its downfall once networks got fast — dataflow engines (Spark/Flink) beat it by keeping intermediates in memory and tracking recomputation via lineage or checkpoints.
5. **Shuffle is the foundational distributed primitive.** Once you have a shuffle, you have sort-merge joins, group-by aggregations, secondary sorts — all the SQL-style operators fall out of it.
6. **High-level APIs (SQL, DataFrames) unlock optimization.** A cost-based optimizer can reorder joins, pick broadcast-vs-sort-merge, and exploit columnar formats in ways hand-written MapReduce never could.
7. **Batch outputs reach production via streams or bulk-loaded files**, never by per-record writes from inside parallel tasks — that's the one consistent anti-pattern the chapter explicitly warns against.

## See Also

- [[01-unix-tools-as-batch-foundation]]
- [[02-distributed-storage-for-batch]]
- [[03-distributed-job-orchestration]]
- [[04-mapreduce]]
- [[05-dataflow-engines-spark-flink]]
- [[06-shuffle-and-distributed-joins]]
- [[07-serving-derived-data-from-batch]]